
# Loan Default Analysis: Exploratory Data Analysis

## 1. Business Problem & Project Objective

Loan defaults create financial losses for lenders and make credit-risk assessment an important part of lending operations. Understanding the characteristics associated with default can help financial institutions evaluate borrower risk and investigate potential improvements to their underwriting process.

The objective of this project is to perform Exploratory Data Analysis (EDA) on a loan dataset to identify patterns and relationships between borrower characteristics, loan attributes, and loan default outcomes.

### Key Questions
- What proportion of loans result in default?
- How do borrower characteristics differ between defaulted and repaid loans?
- Which numerical and categorical features are associated with higher default rates?
- Are there meaningful relationships between financial variables and loan default?
- Can borrower segments be used to explore differences in observed credit risk?

### Analytical Approach

This analysis uses Python, Pandas, NumPy, Matplotlib, Seaborn, and SciPy. The workflow includes data inspection, data-quality assessment, exploratory visualization, feature engineering, statistical hypothesis testing, Weight of Evidence (WoE), Information Value (IV), and risk-segment analysis.




## 2. Dataset Loading and Initial Overview

The first step is to load the loan dataset and understand its structure before conducting detailed analysis. This includes reviewing the number of observations, available features, data types, and sample records.

The dataset contains **255,347 rows and 18 columns**, providing a substantial number of loan records for exploratory analysis.

The variables cover borrower demographics, income, credit history indicators, loan characteristics, and the target variable `Default`.

This initial inspection helps identify the types of analyses that can be performed and highlights the need to verify data quality before interpreting the results.


In [ ]:
# ============================================================
# SECTION 2: Environment Setup & Data Loading
# ============================================================
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Plot styling
sns.set_theme(style="whitegrid")
plt.rcParams["font.size"] = 10
plt.rcParams["figure.titlesize"] = 14

# Dataset path configuration:
# 1. Set LOAN_DEFAULT_DATA_PATH in your environment, or
# 2. Update DATA_PATH below to the location of Loan_default.csv.
# The notebook checks common local and Kaggle locations if the default is absent.
_default_candidates = [
    Path("data/Loan_default.csv"),
    Path("Loan_default.csv"),
    Path("/kaggle/input/datasets/debkumarmondal/loan-default-dataset/Loan_default.csv"),
]
_env_data_path = os.getenv("LOAN_DEFAULT_DATA_PATH")
DATA_PATH = Path(_env_data_path).expanduser() if _env_data_path else next(
    (candidate for candidate in _default_candidates if candidate.exists()),
    _default_candidates[0],
)

if not DATA_PATH.is_file():
    candidate_list = "\n".join(f"  - {path}" for path in _default_candidates)
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH}\n"
        "Set LOAN_DEFAULT_DATA_PATH or update DATA_PATH to your CSV file.\n"
        f"Common locations checked by default:\n{candidate_list}"
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {DATA_PATH}")
print(f"Dataset shape: {df.shape[0]:,} rows | {df.shape[1]:,} columns")
display(df.head())
df.info()


## 3. Data Quality Assessment

Before analyzing loan default patterns, I assessed the dataset's column data types and checked for missing values.

The missing-value assessment reports **zero missing values across all 18 columns**. Therefore, the current dataset does not require missing-value imputation based on this check.

However, missing-value completeness alone does not guarantee data quality. Duplicate records, duplicate loan identifiers, invalid values, and inconsistent categories should also be checked before treating the dataset as fully validated.

This step establishes the initial data-quality baseline for the remaining analysis.


In [ ]:
# ============================================================
# SECTION 3: Data Audit & Missing Value Check
# ============================================================
# 1. Column info and types
print("--- Column Types ---")
display(df.dtypes.value_counts())

# 2. Missing values assessment
missing_summary = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing (%)': (df.isnull().sum() / len(df)) * 100
}).sort_values(by='Missing Count', ascending=False)

missing_cols = missing_summary[missing_summary['Missing Count'] > 0]
print(f"\nTotal columns with missing values: {len(missing_cols)}")
display(missing_cols.head(10))

# 3. Visualize top missing features
if not missing_cols.empty:
    plt.figure(figsize=(10, 4))
    sns.barplot(
        x=missing_cols.head(10)['Missing (%)'],
        y=missing_cols.head(10).index,
        palette='crest'
    )
    plt.title('Top 10 Columns by Missing Value Percentage')
    plt.xlabel('Percentage Missing (%)')
    plt.show()


## 4. Loan Default Distribution

The target variable, `Default`, identifies whether a loan resulted in default. In this analysis, `0` represents a non-defaulted loan and `1` represents a defaulted loan.

The dataset contains:

- **225,694 non-defaulted loans**, representing approximately 88.39% of records.
- **29,653 defaulted loans**, representing approximately 11.61% of records.

The overall default rate provides a baseline against which default rates across different borrower segments can be compared.

The target distribution is imbalanced because non-defaulted loans substantially outnumber defaulted loans. This should be considered when interpreting subsequent comparisons and when evaluating any future predictive model.


In [ ]:
# ============================================================
# SECTION 4: Target Variable Distribution
# ============================================================
target_col = 'Default'

# Convert to integer 0 and 1 (if not already)
df['is_default'] = df[target_col].astype(int)

# Check counts and default rate
default_counts = df['is_default'].value_counts()
default_rate = df['is_default'].mean() * 100

print(f"Target Column: '{target_col}'")
print(f"Overall Default Rate: {default_rate:.2f}%\n")
print(default_counts)

# Plot class balance
plt.figure(figsize=(6, 4))
ax = sns.barplot(x=default_counts.index, y=default_counts.values, palette=['#1f77b4', '#d62728'])
plt.title(f'Loan Outcome Distribution (Default Rate: {default_rate:.2f}%)', fontsize=12, pad=10)
plt.xlabel('Default Status (0 = Repaid, 1 = Defaulted)')
plt.ylabel('Number of Borrowers')
plt.xticks([0, 1], ['Repaid (0)', 'Defaulted (1)'])

for p in ax.patches:
    h = p.get_height()
    ax.annotate(f'{h:,}\n({h/len(df):.1%})', 
                (p.get_x() + p.get_width() / 2., h / 2),
                ha='center', va='center', color='white', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()


## 5. Numerical Feature Analysis

This section compares numerical borrower and loan characteristics between defaulted and non-defaulted loans. Median values are used to summarize the typical observation in each group, while box plots help visualize differences in distributions.

### Key Observations

- **Age:** The median age is 34 among defaulted borrowers, compared with 45 among non-defaulted borrowers.
- **Income:** Defaulted borrowers have a lower median income of 66,566, compared with 84,237.50.
- **Loan amount:** The median loan amount is higher among defaulted borrowers (152,672 versus 124,236).
- **Interest rate:** The median interest rate is 16.93% for defaulted loans and 12.99% for non-defaulted loans.
- **Employment tenure:** Defaulted borrowers have a lower median employment tenure of 46 months, compared with 61 months.
- **Credit score:** The median credit score is 553 among defaulted borrowers and 577 among non-defaulted borrowers.

These differences suggest that borrower age, income, loan size, interest rate, credit score, and employment tenure warrant further investigation. They describe associations in the observed dataset and do not independently establish default causation.


In [ ]:
# ============================================================
# SECTION 5: Numerical Feature Analysis
# ============================================================
num_cols = ['Age', 'Income', 'LoanAmount', 'CreditScore', 'InterestRate', 'DTIRatio', 'MonthsEmployed']

# 1. Median comparison table
print("--- Median Values by Loan Outcome ---")
display(df.groupby('is_default')[num_cols].median().T.rename(columns={0: 'Repaid (Median)', 1: 'Defaulted (Median)'}))

# 2. Box plots (with outlier points hidden for clean viewing)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, col in enumerate(num_cols):
    sns.boxplot(
        x='is_default', 
        y=col, 
        data=df, 
        ax=axes[idx], 
        palette=['#1f77b4', '#d62728'], 
        showfliers=False
    )
    axes[idx].set_title(f'{col} vs Default', fontweight='bold')
    axes[idx].set_xlabel('')
    axes[idx].set_xticklabels(['Repaid', 'Defaulted'])

# Remove the unused 8th subplot
fig.delaxes(axes[7])
plt.tight_layout()
plt.show()


## 6. Default Rates Across Categorical Features

Numerical variables do not capture every aspect of a borrower's profile. This section compares observed default rates across categorical features, including education, employment type, marital status, loan purpose, mortgage status, dependent status, and co-signer status.

For each category, the default rate is calculated as the number of defaulted loans divided by the total number of loans in that category.

The bar charts help identify categories with relatively higher or lower observed default rates. However, a higher rate does not necessarily mean that the category itself causes default. Differences in borrower characteristics, group sizes, and interactions with other variables may also contribute to the pattern.

These comparisons provide a starting point for more detailed risk-segment analysis.


In [ ]:
# ============================================================
# SECTION 6: Categorical Feature Default Rates
# ============================================================
cat_cols = ['Education', 'EmploymentType', 'MaritalStatus', 'LoanPurpose', 'HasMortgage', 'HasDependents', 'HasCoSigner']

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, col in enumerate(cat_cols):
    # Calculate default percentage per category
    group = df.groupby(col)['is_default'].agg(['count', 'mean']).reset_index()
    group['default_rate_%'] = group['mean'] * 100
    group = group.sort_values(by='default_rate_%', ascending=False)
    
    sns.barplot(x=col, y='default_rate_%', data=group, ax=axes[idx], palette='flare')
    axes[idx].set_title(f'Default Rate by {col}', fontweight='bold')
    axes[idx].set_ylabel('Default Rate (%)')
    axes[idx].set_xlabel('')
    axes[idx].tick_params(axis='x', rotation=25)
    
    for p in axes[idx].patches:
        axes[idx].annotate(f"{p.get_height():.1f}%", 
                           (p.get_x() + p.get_width() / 2., p.get_height() + 0.3),
                           ha='center', fontsize=9)

# Clean up empty subplots
fig.delaxes(axes[7])
fig.delaxes(axes[8])
plt.tight_layout()
plt.show()


## 7. Correlation Analysis

A Pearson correlation heatmap is used to examine linear relationships among numerical variables and their associations with the loan default indicator.

### Key Observations

The strongest negative correlations with the default indicator among the reported numerical features are:

- **Age:** approximately -0.168
- **Income:** approximately -0.099
- **Months employed:** approximately -0.097

Interest rate has a positive correlation of approximately 0.131 with default, while loan amount has a positive correlation of approximately 0.087.

These correlations are relatively modest in magnitude. This suggests that no single numerical variable should be assumed to explain loan default on its own.

Correlation measures association rather than causation. It may also miss nonlinear relationships and interactions between variables.


In [ ]:
# ============================================================
# SECTION 7: Correlation Heatmap
# ============================================================
numeric_features = df.select_dtypes(include=['int64', 'float64']).columns.drop(['LoanID'], errors='ignore')

plt.figure(figsize=(10, 8))
corr = df[numeric_features].corr()

sns.heatmap(
    corr, 
    annot=True, 
    fmt='.2f', 
    cmap='vlag', 
    center=0, 
    linewidths=0.5,
    cbar_kws={'label': 'Pearson Correlation'}
)
plt.title('Correlation Heatmap of Financial & Demographic Features', pad=12)
plt.show()

print("\n--- Features Most Correlated with Default ---")
display(corr['is_default'].drop('is_default').sort_values(ascending=False))


## 8. Feature Engineering: Financial Ratios and Borrower Tiers

This section creates additional features to make borrower-level comparisons easier to interpret.

### Derived Features

- **Loan-to-Income (LTI):** Loan amount divided by annual income, with a small constant added to the denominator to avoid division by zero.
- **Monthly Income:** Annual income divided by 12.
- **Credit Score Tier:** Credit scores grouped into predefined ranges for comparative analysis.
- **Employment Tenure:** Months employed grouped into employment-duration categories.

Default rates are then compared across credit-score tiers and employment-tenure groups.

These derived categories make it easier to identify broad patterns than examining every individual score or tenure value separately.

The credit-score boundaries used here are analytical groupings and should not automatically be interpreted as official FICO classifications. Similarly, the observed default rates across these tiers do not establish causation.


In [ ]:
# ============================================================
# SECTION 8: Financial Ratio Engineering & Credit Tiers
# ============================================================
# 1. Leverage & Burden Ratios
df['LoanToIncome'] = df['LoanAmount'] / (df['Income'] + 1)
df['MonthlyIncome'] = df['Income'] / 12

# 2. Industry Standard FICO Credit Score Tiers
bins_credit = [0, 579, 669, 739, 799, 900]
labels_credit = ['Poor (<580)', 'Fair (580-669)', 'Good (670-739)', 'Very Good (740-799)', 'Exceptional (800+)']
df['CreditTier'] = pd.cut(df['CreditScore'], bins=bins_credit, labels=labels_credit)

# 3. Employment Longevity Tiers
bins_emp = [-1, 12, 36, 60, 120, 500]
labels_emp = ['< 1 Year', '1-3 Years', '3-5 Years', '5-10 Years', '10+ Years']
df['EmploymentTenure'] = pd.cut(df['MonthsEmployed'], bins=bins_emp, labels=labels_emp)

# Visualizing Default Rates Across Derived Tiers
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

# Credit Tier Plot
credit_perf = df.groupby('CreditTier', observed=False)['Default'].agg(['count', 'mean']).reset_index()
credit_perf['DefaultRate%'] = credit_perf['mean'] * 100
sns.barplot(x='CreditTier', y='DefaultRate%', data=credit_perf, ax=axes[0], palette='Blues_r')
axes[0].set_title('Default Rate by Credit Score Tier', fontweight='bold')
axes[0].set_ylabel('Default Rate (%)')
axes[0].tick_params(axis='x', rotation=25)
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() + 0.3), ha='center')

# Employment Tenure Plot
emp_perf = df.groupby('EmploymentTenure', observed=False)['Default'].agg(['count', 'mean']).reset_index()
emp_perf['DefaultRate%'] = emp_perf['mean'] * 100
sns.barplot(x='EmploymentTenure', y='DefaultRate%', data=emp_perf, ax=axes[1], palette='Purples_r')
axes[1].set_title('Default Rate by Employment Tenure', fontweight='bold')
axes[1].set_ylabel('Default Rate (%)')
axes[1].tick_params(axis='x', rotation=25)
for p in axes[1].patches:
    axes[1].annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() + 0.3), ha='center')

plt.tight_layout()
plt.show()


## 9. Statistical Hypothesis Testing

Visual comparisons can reveal patterns, but statistical tests help assess whether observed differences are compatible with the assumption of no association.

### Numerical Features: Mann–Whitney U Test

The Mann–Whitney U test compares the distributions of numerical features between defaulted and non-defaulted loans.

- **Null hypothesis (H₀):** The two groups have the same underlying distribution for the feature being tested.
- **Alternative hypothesis (H₁):** The distributions differ between the two groups.

### Categorical Features: Chi-Square Test

The chi-square test of independence examines whether a categorical feature is associated with the default outcome.

- **Null hypothesis (H₀):** The categorical feature and default outcome are independent.
- **Alternative hypothesis (H₁):** An association exists between the feature and default outcome.

The notebook uses a significance threshold of 0.01. Very small p-values indicate evidence against the corresponding null hypothesis, but statistical significance does not establish practical importance or causation.

Because the dataset is large and multiple features are tested, effect sizes and multiple-testing adjustments should also be considered when interpreting the results.


In [ ]:
# ============================================================
# SECTION 9: Formal Statistical Hypothesis Testing
# ============================================================
from scipy import stats
from statsmodels.stats.multitest import multipletests

def format_p_value(p_value):
    """Format tiny/underflowed p-values without implying they are exactly zero."""
    if p_value is None or not np.isfinite(p_value):
        return "not estimable"
    if p_value == 0:
        return "p < 1e-300 (numerical underflow)"
    if p_value < 1e-4:
        return f"p = {p_value:.2e}"
    return f"p = {p_value:.4f}"

print("=== 1. NUMERICAL FEATURES: MANN-WHITNEY U TEST ===")
print("Effect size: rank-biserial correlation (range -1 to +1).")
num_test_cols = [
    'Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed',
    'InterestRate', 'DTIRatio', 'LoanToIncome'
]
num_test_cols = [c for c in num_test_cols if c in df.columns]

num_hyp_results = []
for col in num_test_cols:
    sample_repaid = pd.to_numeric(df.loc[df['Default'] == 0, col], errors='coerce').dropna()
    sample_default = pd.to_numeric(df.loc[df['Default'] == 1, col], errors='coerce').dropna()

    if len(sample_repaid) == 0 or len(sample_default) == 0:
        continue

    test = stats.mannwhitneyu(
        sample_repaid, sample_default, alternative='two-sided', method='asymptotic'
    )
    n_repaid, n_default = len(sample_repaid), len(sample_default)
    # SciPy's U is for the first sample (repaid); this sign convention makes
    # positive values indicate larger values in the repaid group.
    rank_biserial = (2 * test.statistic / (n_repaid * n_default)) - 1

    num_hyp_results.append({
        'Feature': col,
        'Repaid n': n_repaid,
        'Defaulted n': n_default,
        'Repaid Median': round(sample_repaid.median(), 2),
        'Defaulted Median': round(sample_default.median(), 2),
        'Rank-biserial r': round(rank_biserial, 4),
        'P-Value': format_p_value(test.pvalue),
        '_p_raw': test.pvalue
    })

num_hyp_df = pd.DataFrame(num_hyp_results)
if not num_hyp_df.empty:
    # Benjamini-Hochberg controls the false-discovery rate across tests in this family.
    valid_p = num_hyp_df['_p_raw'].fillna(1).to_numpy()
    num_hyp_df['FDR-adjusted p'] = [
        format_p_value(p) for p in multipletests(valid_p, method='fdr_bh')[1]
    ]
    num_hyp_df['Significant (FDR q < 0.01)'] = [
        'Yes' if q < 0.01 else 'No'
        for q in multipletests(valid_p, method='fdr_bh')[1]
    ]
    display(num_hyp_df.drop(columns=['_p_raw']))

print("\\n=== 2. CATEGORICAL FEATURES: CHI-SQUARE TEST OF INDEPENDENCE ===")
print("Effect size: Cramer's V (0 to 1; larger values indicate stronger association).")
cat_test_cols = [
    'Education', 'EmploymentType', 'MaritalStatus', 'LoanPurpose',
    'HasMortgage', 'HasDependents', 'HasCoSigner', 'LoanTerm'
]
cat_test_cols = [c for c in cat_test_cols if c in df.columns]

cat_hyp_results = []
for col in cat_test_cols:
    contingency = pd.crosstab(df[col], df['Default'])
    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
        continue

    chi2, p_val, dof, expected = stats.chi2_contingency(contingency)
    n = contingency.to_numpy().sum()
    min_dimension = min(contingency.shape[0] - 1, contingency.shape[1] - 1)
    cramers_v = np.sqrt(chi2 / (n * min_dimension)) if n > 0 and min_dimension > 0 else np.nan

    cat_hyp_results.append({
        'Categorical Feature': col,
        'N': int(n),
        'Chi2 Score': round(chi2, 2),
        'Degrees of Freedom': dof,
        "Cramer's V": round(cramers_v, 4),
        'P-Value': format_p_value(p_val),
        '_p_raw': p_val
    })

cat_hyp_df = pd.DataFrame(cat_hyp_results)
if not cat_hyp_df.empty:
    cat_q = multipletests(cat_hyp_df['_p_raw'].fillna(1).to_numpy(), method='fdr_bh')[1]
    cat_hyp_df['FDR-adjusted p'] = [format_p_value(p) for p in cat_q]
    cat_hyp_df['Significant (FDR q < 0.01)'] = ['Yes' if q < 0.01 else 'No' for q in cat_q]
    display(cat_hyp_df.drop(columns=['_p_raw']))

print("\\nInterpretation notes:")
print("- A very small p-value is evidence against the null hypothesis, not a measure of effect size.")
print("- Rank-biserial correlation is signed: positive means the repaid group tends to have larger values.")
print("- Cramer's V is non-directional; interpret its magnitude alongside the contingency table and sample sizes.")
print("- FDR-adjusted p-values account for multiple comparisons within each feature family.")



## 10. Weight of Evidence (WoE) and Information Value (IV)

Weight of Evidence (WoE) and Information Value (IV) are commonly used in credit-risk analysis to evaluate how a feature separates non-defaulted and defaulted observations.

For this analysis, numerical features with many unique values are divided into quantile-based bins, while categorical features are evaluated by category.

Information Value is calculated from the distributions of non-defaulted and defaulted loans across the feature's groups. A higher IV generally indicates greater univariate separation within the analyzed dataset.

### Key Observations

The calculated IV ranking identifies the following features as having the highest reported values:

- **Age:** 0.2875
- **Loan-to-Income:** 0.1987
- **Interest Rate:** 0.1731
- **Income:** 0.1243

These results suggest that these features provide relatively stronger separation between the two loan-outcome groups under the current binning and calculation approach.

IV is not proof of causal influence or validated predictive performance. The current implementation also excludes bins with zero good or bad observations, which can affect the resulting score. A smoothing method should be considered before using these values for formal credit-risk modeling.


In [ ]:
# ============================================================
# SECTION 10: Weight of Evidence (WoE) & Information Value (IV)
# ============================================================
def get_iv(data, feature, target='Default', n_bins=10, smoothing=0.5, return_table=False):
    """
    Calculate WoE and Information Value using additive smoothing.

    target convention: 0 = good/repaid, 1 = bad/defaulted.
    Numeric features with more than 10 distinct values are quantile-binned.
    Missing feature/target rows are excluded; smoothing avoids zero proportions.
    """
    if feature not in data.columns or target not in data.columns:
        raise KeyError(f"Missing feature or target column: {feature!r} / {target!r}")

    temp = data[[feature, target]].dropna().copy()
    temp = temp[temp[target].isin([0, 1])]
    if temp.empty:
        return (np.nan, pd.DataFrame()) if return_table else np.nan

    # Quantile-bin continuous/high-cardinality numeric variables.
    if pd.api.types.is_numeric_dtype(temp[feature]) and temp[feature].nunique() > 10:
        try:
            temp['_iv_bin'] = pd.qcut(temp[feature], q=n_bins, duplicates='drop')
            bin_col = '_iv_bin'
        except ValueError:
            # Fall back to unique-value categories if quantile binning is not feasible.
            temp['_iv_bin'] = temp[feature].astype('string')
            bin_col = '_iv_bin'
    else:
        bin_col = feature

    grouped = temp.groupby(bin_col, observed=False, dropna=False)[target].agg(
        goods=lambda s: (s == 0).sum(),
        bads=lambda s: (s == 1).sum()
    ).astype(float)

    # Remove empty categorical bins, if any, before calculating distributions.
    grouped = grouped[(grouped['goods'] + grouped['bads']) > 0].copy()
    if grouped.empty:
        return (np.nan, grouped) if return_table else np.nan

    k = len(grouped)
    total_goods = grouped['goods'].sum()
    total_bads = grouped['bads'].sum()
    if total_goods == 0 or total_bads == 0:
        return (np.nan, grouped) if return_table else np.nan

    # Additive (Laplace-style) smoothing applied consistently to each bin.
    grouped['dist_good'] = (grouped['goods'] + smoothing) / (total_goods + smoothing * k)
    grouped['dist_bad'] = (grouped['bads'] + smoothing) / (total_bads + smoothing * k)
    grouped['WoE'] = np.log(grouped['dist_good'] / grouped['dist_bad'])
    grouped['IV_component'] = (
        grouped['dist_good'] - grouped['dist_bad']
    ) * grouped['WoE']
    iv_value = grouped['IV_component'].sum()

    return (iv_value, grouped) if return_table else iv_value


candidate_features = [
    'Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed',
    'InterestRate', 'DTIRatio', 'LoanToIncome', 'Education',
    'EmploymentType', 'HasCoSigner', 'HasMortgage', 'LoanPurpose'
]
candidate_features = [c for c in candidate_features if c in df.columns]

iv_list = []
for feat in candidate_features:
    score = get_iv(df, feat)
    if not np.isfinite(score):
        tier = "Not estimable"
    elif score < 0.02:
        tier = "Very weak"
    elif score < 0.1:
        tier = "Weak"
    elif score < 0.3:
        tier = "Medium"
    else:
        tier = "Strong (review for overfitting/leakage)"
    iv_list.append({
        'Feature': feat,
        'IV': round(score, 4) if np.isfinite(score) else np.nan,
        'Indicative IV Band': tier
    })

iv_ranking_df = pd.DataFrame(iv_list).sort_values(by='IV', ascending=False, na_position='last')

plt.figure(figsize=(9, 4.5))
sns.barplot(x='IV', y='Feature', data=iv_ranking_df, hue='Feature', legend=False, palette='viridis')
plt.title('Feature Separation by Information Value (IV)', fontweight='bold')
plt.axvline(0.02, color='gray', linestyle='--', label='0.02')
plt.axvline(0.10, color='orange', linestyle='--', label='0.10')
plt.axvline(0.30, color='red', linestyle='--', label='0.30')
plt.legend(loc='lower right', title='Reference thresholds')
plt.tight_layout()
plt.show()

display(iv_ranking_df)

print(
    "IV caveats: these are in-sample, univariate separation measures. "
    "Values depend on binning and smoothing, do not establish causality or out-of-sample "
    "predictive performance, and should be checked for leakage and stability."
)



## 11. Two-Dimensional Risk Interaction Analysis

Individual feature analysis does not show how two characteristics may relate to default simultaneously. This section uses heatmaps to compare default rates across combinations of selected features.

### Analysis 1: Co-Signer Status and Credit Score Tier

The first heatmap compares default rates across credit-score tiers and co-signer status. It helps investigate whether observed default rates differ between borrowers with and without a co-signer within each score tier.

### Analysis 2: Interest Rate and Loan Term

The second heatmap compares default rates across interest-rate brackets and loan terms. It helps examine whether the observed relationship between interest rates and default varies across loan durations.

The heatmaps provide descriptive comparisons of joint segments. Group counts should be considered alongside default rates, and the observed patterns should not be interpreted as proof that a co-signer, interest rate, or loan term independently causes a change in default risk.


In [ ]:
# ============================================================
# SECTION 11: 2D Risk Interaction Heatmaps
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Co-Signer Protection Effect across Credit Tiers
if 'HasCoSigner' in df.columns and 'CreditTier' in df.columns:
    cosigner_pivot = df.pivot_table(
        index='CreditTier', 
        columns='HasCoSigner', 
        values='Default', 
        aggfunc='mean',
        observed=False
    ) * 100
    
    sns.heatmap(cosigner_pivot, annot=True, fmt='.1f', cmap='coolwarm', cbar_kws={'label': 'Default Rate (%)'}, ax=axes[0])
    axes[0].set_title('Co-Signer Impact across Credit Tiers (%)', fontweight='bold')
    axes[0].set_ylabel('Credit Score Tier')
    axes[0].set_xlabel('Has Co-Signer')

# 2. Term vs Interest Rate Interaction
if 'LoanTerm' in df.columns and 'InterestRate' in df.columns:
    df['Rate_Bracket'] = pd.qcut(df['InterestRate'], q=4, labels=['Low Rate', 'Medium Rate', 'High Rate', 'Very High Rate'])
    
    term_rate_pivot = df.pivot_table(
        index='Rate_Bracket', 
        columns='LoanTerm', 
        values='Default', 
        aggfunc='mean',
        observed=False
    ) * 100
    
    sns.heatmap(term_rate_pivot, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': 'Default Rate (%)'}, ax=axes[1])
    axes[1].set_title('Loan Term vs Interest Rate Default Rate (%)', fontweight='bold')
    axes[1].set_ylabel('Interest Rate Bracket')
    axes[1].set_xlabel('Loan Term (Months)')

plt.tight_layout()
plt.show()


## 12. Conclusion and Business Recommendations

This project explored loan default patterns using borrower demographics, financial characteristics, loan attributes, statistical hypothesis testing, feature engineering, and risk-segment analysis.

### Key Findings

- The overall observed loan default rate is **11.61%**.
- Defaulted borrowers have lower median age, income, credit score, and employment tenure in the reported comparisons.
- Defaulted loans have higher median loan amounts and interest rates.
- Information Value analysis identifies age, loan-to-income ratio, interest rate, and income among the strongest univariate separators in the current analysis.
- The DTI–LTI matrix provides an additional way to compare default rates across financial-burden segments.

### Business Implications

Lenders could investigate the borrower characteristics and financial segments associated with higher observed default rates when reviewing credit-risk assessment practices. These findings are exploratory and should not be used as standalone loan-approval rules.

### Limitations

- The analysis identifies associations and does not establish causation.
- Statistical significance does not necessarily indicate practical importance.
- The Information Value results depend on feature binning and calculation choices.
- Further data validation, fairness assessment, and out-of-sample testing would be needed before developing a predictive credit-risk model.

### Future Scope

Future work could include building a classification model, comparing evaluation metrics, assessing probability calibration, and evaluating the financial impact of different lending decisions.


In [ ]:
# ============================================================
# SECTION 12: Underwriting Decision Matrix
# ============================================================

# Create risk categories using the existing columns
df['DTI_Risk'] = pd.qcut(
    df['DTIRatio'],
    q=4,
    labels=['Low DTI', 'Moderate DTI', 'High DTI', 'Critical DTI']
)

df['LTI_Risk'] = pd.qcut(
    df['LoanToIncome'],
    q=4,
    labels=['Low LTI', 'Moderate LTI', 'High LTI', 'Critical LTI']
)

# Calculate loan volume and default rate
underwriting_pivot = df.pivot_table(
    index='DTI_Risk',
    columns='LTI_Risk',
    values='Default',
    aggfunc=['count', 'mean'],
    observed=False
)

# Convert default rates into percentages
underwriting_pivot['mean'] = underwriting_pivot['mean'] * 100

# Rename columns for readability
underwriting_pivot.columns = [
    f"{stat} - {col}"
    for stat, col in underwriting_pivot.columns
]

# Display the underwriting matrix
print("--- Credit Risk Underwriting Matrix ---")
display(underwriting_pivot.round(2))

# Visualize default rates across DTI and LTI categories
default_matrix = df.pivot_table(
    index='DTI_Risk',
    columns='LTI_Risk',
    values='Default',
    aggfunc='mean',
    observed=False
) * 100

plt.figure(figsize=(10, 6))

sns.heatmap(
    default_matrix,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    linewidths=0.5,
    cbar_kws={'label': 'Default Rate (%)'}
)

plt.title(
    'Loan Default Risk: DTI vs LTI',
    fontsize=14,
    fontweight='bold'
)
plt.xlabel('Loan-to-Income (LTI) Risk')
plt.ylabel('Debt-to-Income (DTI) Risk')
plt.tight_layout()
plt.show()